# Entrenamiento del detector de baches (Urban Road Monitor)

Este cuaderno corre en **Google Colab** con GPU (Entorno de ejecución → Cambiar tipo → T4).
Hace, en orden: instalar, traer el dataset, revisarlo, prepararlo, entrenar, evaluar y exportar.

Antes de empezar:
1. Descarga cada dataset desde Roboflow Universe en formato **YOLOv8** (botón *Download Dataset*, *zip*).
2. Súbelos a tu Google Drive, en la carpeta `urban-road-monitor/datasets/` (un `.zip` por dataset).
3. Anota la licencia de cada uno: queda en el archivo `data.yaml` si Roboflow la declara.

No pegues claves ni contraseñas en este cuaderno.

In [ ]:
# 1. Traer el código de entrenamiento. Si el repositorio es privado, sube la carpeta `training/`
#    como .zip desde el panel de archivos de Colab y descomprímela en /content/repo.
REPO_URL = "https://github.com/arleyriascos/urban-road-monitor-ai"
!git clone {REPO_URL} /content/repo
%cd /content/repo

In [ ]:
# 2. Instalar Ultralytics en la versión fija del proyecto.
!pip install -q -r training/requirements.txt

In [ ]:
# 3. Conectar Drive y descomprimir los datasets en training/data/raw/<nombre>.
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATASETS_DIR = Path("/content/drive/MyDrive/urban-road-monitor/datasets")
RAW_DIR = Path("training/data/raw")
for zip_path in sorted(DATASETS_DIR.glob("*.zip")):
    target = RAW_DIR / zip_path.stem.replace(" ", "_")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)
    print(zip_path.name, "->", target)
print(sorted(path.name for path in RAW_DIR.iterdir()))

In [ ]:
# 4. Revisar los datasets sin escribir nada.
#    Si algo sale mal, aquí se ve cuál archivo y cuál línea.
import glob

SOURCES = sorted(glob.glob("training/data/raw/*"))
args = " ".join(f"--source {s}" for s in SOURCES)
!python -m training.prepare {args} --output training/data/pothole --audit-only

In [ ]:
# 5. Preparar el dataset único: fotos agrupadas, 80/10/10 y una sola clase (pothole).
!rm -rf training/data/pothole
!python -m training.prepare {args} --output training/data/pothole

In [ ]:
# 6. Entrenar (YOLOv8n, 416 px, 50 épocas). Tarda de 30 a 90 minutos en una T4.
!python -m training.train --data training/data/pothole/data.yaml --device 0

In [ ]:
# 7. Medir con el conjunto de prueba, que el modelo nunca vio.
!python -m training.evaluate \
  --weights training/runs/pothole/weights/best.pt \
  --data training/data/pothole/data.yaml

In [ ]:
# 8. Exportar para el navegador. ONNX funciona con cualquier versión de Ultralytics.
#    Para TensorFlow.js usa --format tfjs (solo con la versión fija de
#    requirements.txt y Python < 3.13).
!python -m training.export \
  --weights training/runs/pothole/weights/best.pt --format onnx

In [ ]:
# 9. Guardar en Drive los pesos, el modelo exportado, las métricas y la lista de datasets usados.
OUT = "/content/drive/MyDrive/urban-road-monitor/modelo"
!mkdir -p {OUT}
!cp training/runs/pothole/weights/best.pt training/runs/pothole/weights/best.onnx {OUT}/
!cp training/runs/pothole/weights/model_info.json {OUT}/
!cp training/runs/pothole/metrics.json {OUT}/
!cp training/data/pothole/manifest.json {OUT}/
!ls -la {OUT}

Cuando termine, copia a Arley `best.onnx`, `model_info.json`, `metrics.json` y `manifest.json`.
Con esas métricas y las licencias del manifiesto se completa la ficha del modelo (`training/MODEL_CARD.md`).